# KK1 – Analys av hundraser

Detta dataset beskriver **117 hundraser**, en rad per ras, med 8 kolumner: ras, ursprungsland, pälsfärg, höjd, ögonfärg, livslängd, karaktärsdrag och vanliga hälsoproblem.

**Källa:** Kaggle – https://www.kaggle.com/datasets/marshuu/dog-breeds

Höjden anges i **tum** i källan (kolumnen `Height (in)`) och livslängden i år, båda som textintervall (t.ex. `"21-24"`). Vi parsar dem till siffror och **räknar om höjden till centimeter**, eftersom rapporten är på svenska.

I notebooken läser vi in datan, inspekterar den, tvättar de två intervallkolumnerna och utforskar mönster med tre visualiseringar.

## Inläsning och inspektion

Först en mekanisk översikt av datan – form, kolumner, datatyper och saknade värden – innan vi tvättar något.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Läs in datasetet
df = pd.read_csv("data/dog_breeds.csv")
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all")

**Observationer:**

- Datasetet har 117 rader och 8 kolumner.
- Alla kolumner läses in som text – ingen är numerisk ännu.
- `df.info()` visar inga saknade värden i någon kolumn.
- `Height (in)` och `Longevity (yrs)` är textintervall (t.ex. `"21-24"`) och måste parsas till siffror för att kunna plottas. Det gör vi i nästa sektion.

## Datatvätt

`Height (in)` och `Longevity (yrs)` är textintervall (t.ex. `"21-24"`) och går inte att plotta som de är. Vi tvättar dem medvetet:

- Först kontrollerar vi saknade värden i stället för att blint köra `dropna()`.
- Varje intervall ersätts med sin **mittpunkt** – ett enda tal som går att plotta. Mittpunkten döljer spridningen inom intervallet; vi återkommer till det i avslutningen.
- Höjden räknas om från **tum till centimeter** (× 2,54, avrundat till heltal) eftersom rapporten är på svenska.

In [ ]:
# Medveten kontroll av saknade värden innan vi tvättar
df.isna().sum()

In [ ]:
def range_mitt(text):
    """Tar ett intervall som '21-24' och returnerar mittpunkten (22.5)."""
    low, high = text.split("-")
    return (int(low) + int(high)) / 2

# Höjd: tum -> cm, avrundat till heltal
df["Höjd_cm"] = (df["Height (in)"].apply(range_mitt) * 2.54).round(0).astype(int)

# Livslängd: mittpunkt i år
df["Livslängd_mitt"] = df["Longevity (yrs)"].apply(range_mitt)

df[["Breed", "Height (in)", "Höjd_cm", "Longevity (yrs)", "Livslängd_mitt"]].head()

In [ ]:
# Verifiera att de nya kolumnerna är numeriska och utan saknade värden
print(df[["Höjd_cm", "Livslängd_mitt"]].dtypes)
print("Saknade värden i nya kolumner:", int(df[["Höjd_cm", "Livslängd_mitt"]].isna().sum().sum()))

**Resultat av tvätten:**

- Inga saknade värden fanns, så inga rader behövde tas bort – ett medvetet konstaterande, inte ett blint `dropna()`.
- `Höjd_cm` och `Livslängd_mitt` är nu numeriska och kompletta (117 värden vardera).
- Datan är redo att utforskas med visualiseringar.